# 06 — Customer Prediction, Targeting Policy & Business Impact

## Objective
Turn customer-level heterogeneous treatment-effect estimates into an actionable marketing policy.

This final notebook covers:
- full-customer prediction
- recommended treatment
- targeting depth simulation
- personalized vs fixed/random policies
- incremental conversions
- incremental revenue
- top-customer export
- final portfolio recommendation

In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

# Find the project root so the notebook works whether launched from the project root
# or from inside the notebooks directory.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
ROOT = next((p for p in _candidates if (p / "src" / "config.py").exists()), _here)
if not (ROOT / "src" / "config.py").exists():
    raise FileNotFoundError("Could not locate project root containing src/config.py")

sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
from src.data_utils import load_data, prepare
from src.feature_engineering import make_preprocessor
from src.uplift_models import train_tlearner, predict_tlearner
from src.policy import apply_policy
from src.business_impact import targeting_curve
from src.config import DATA_PATH, RANDOM_SEED, TEST_SIZE, UPLIFT_THRESHOLD, PREDICTIONS_DIR, METRICS_DIR
from sklearn.model_selection import train_test_split

df = prepare(load_data(DATA_PATH))
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE,
    random_state=RANDOM_SEED, stratify=df["segment"]
)
train = df.iloc[idx_train].reset_index(drop=True)
test = df.iloc[idx_test].reset_index(drop=True)

pre = make_preprocessor()
X_train = pre.fit_transform(train)
X_test = pre.transform(test)
X_all = pre.transform(df)

pred = pd.DataFrame({
    "row_id": np.arange(len(df)),
    "segment": df.segment.to_numpy(),
    "visit": df.visit.to_numpy(),
    "conversion": df.conversion.to_numpy(),
    "spend": df.spend.to_numpy()
})

for arm, slug in [("Mens E-Mail","mens"), ("Womens E-Mail","womens")]:
    mt, mc = train_tlearner(X_train, train.segment.to_numpy(), train.conversion.to_numpy(), arm)
    ft, fc = predict_tlearner(mt, mc, X_all)
    pred[f"pred_conversion_{slug}"] = ft
    pred[f"pred_conversion_control_{slug}"] = fc
    pred[f"uplift_{slug}"] = ft - fc

pred["max_uplift"] = pred[["uplift_mens","uplift_womens"]].max(axis=1)
pred["recommended_action"] = np.where(
    (pred["uplift_mens"] > pred["uplift_womens"]) & (pred["uplift_mens"] > UPLIFT_THRESHOLD),
    "Mens E-Mail",
    np.where(
        (pred["uplift_womens"] > pred["uplift_mens"]) & (pred["uplift_womens"] > UPLIFT_THRESHOLD),
        "Womens E-Mail",
        "No E-Mail"
    )
)
pred["uplift_any_email"] = np.maximum(pred["uplift_mens"], pred["uplift_womens"])

for c in ["recency","history","mens","womens","zip_code","newbie","channel","history_segment"]:
    pred[c] = df[c].values

display(pred.head())

## 1. Customer-level recommendation distribution

In [ ]:
recommendation_counts = pred["recommended_action"].value_counts()
display(recommendation_counts.to_frame("customers"))
display((recommendation_counts / len(pred) * 100).round(2).to_frame("percent"))

## 2. Rank customers by estimated incremental response

In [ ]:
ranked = pred.sort_values("max_uplift", ascending=False).reset_index(drop=True)
ranked["rank"] = np.arange(1, len(ranked)+1)
ranked["targeting_percentile"] = ranked["rank"] / len(ranked)

display(ranked[[
    "rank","recency","history","channel","history_segment",
    "uplift_mens","uplift_womens","recommended_action"
]].head(20))

## 3. Targeting-depth simulator

In [ ]:
def simulate_depth(ranked_df, depth):
    n = max(1, int(len(ranked_df) * depth))
    top = ranked_df.head(n).copy()
    actions = np.where(top["max_uplift"] > UPLIFT_THRESHOLD, top["recommended_action"], "No E-Mail")
    return {
        "depth": depth,
        "customers_targeted": int((actions != "No E-Mail").sum()),
        "mens_emails": int((actions == "Mens E-Mail").sum()),
        "womens_emails": int((actions == "Womens E-Mail").sum()),
        "no_email": int((actions == "No E-Mail").sum()),
        "average_positive_uplift": float(top.loc[actions != "No E-Mail","max_uplift"].mean()) if np.any(actions != "No E-Mail") else 0.0,
    }

depths = [0.05,0.10,0.20,0.30,0.40,0.50,1.00]
depth_table = pd.DataFrame([simulate_depth(ranked, d) for d in depths])
display(depth_table)
depth_table.to_csv(METRICS_DIR / "notebook_06_targeting_depths.csv", index=False)

## 4. Held-out targeting evaluation

In [ ]:
# Evaluate policy using the existing held-out predictions produced by the project's training pipeline.
# If available, use the project's persisted test prediction artifact; otherwise calculate a transparent
# inverse-propensity-weighted estimate from the current test set.
test_pred = pred.iloc[idx_test].copy().reset_index(drop=True)
test_pred["row_id"] = np.arange(len(test_pred))
test_pred["max_uplift"] = test_pred[["uplift_mens","uplift_womens"]].max(axis=1)

def ipw_policy_value(data, actions, outcome):
    probs = data["segment"].value_counts(normalize=True).to_dict()
    total = 0.0
    for i, row in data.iterrows():
        a = actions.iloc[i]
        if row["segment"] == a:
            total += row[outcome] / max(probs.get(a, 0), 1e-9)
    return total / len(data)

def policy_result(data, depth):
    d = data.sort_values("max_uplift", ascending=False).reset_index(drop=True)
    n = max(1, int(len(d) * depth))
    top = d.head(n)
    personalized = np.where(
        top["max_uplift"] > 0, top["recommended_action"], "No E-Mail"
    )
    actions = pd.Series("No E-Mail", index=d.index)
    actions.iloc[:n] = personalized
    value = ipw_policy_value(d, actions, "conversion")
    base = d.loc[d.segment == "No E-Mail", "conversion"].mean()
    return {
        "depth": depth,
        "policy_value": value,
        "baseline_control_conversion": base,
        "incremental_conversion_rate": value - base,
        "estimated_incremental_conversions": (value - base) * len(d)
    }

heldout_depths = pd.DataFrame([policy_result(test_pred, d) for d in depths])
display(heldout_depths.style.format({
    "policy_value":"{:.4%}",
    "baseline_control_conversion":"{:.4%}",
    "incremental_conversion_rate":"{:.4%}",
    "estimated_incremental_conversions":"{:.2f}"
}))
heldout_depths.to_csv(METRICS_DIR / "notebook_06_heldout_targeting.csv", index=False)

## 5. Policy comparison

In [ ]:
def policy_table_local(data, depth=0.20):
    d = data.copy().reset_index(drop=True)
    d["row_id"] = np.arange(len(d))
    order = d.sort_values("max_uplift", ascending=False)
    n = max(1, int(len(d) * depth))
    top_ids = set(order.head(n).row_id)

    policies = {
        "Personalized": np.where(
            d.row_id.isin(top_ids) & (d.max_uplift > 0),
            d.recommended_action, "No E-Mail"
        ),
        "Mens-only": np.where(d.row_id.isin(top_ids), "Mens E-Mail", "No E-Mail"),
        "Womens-only": np.where(d.row_id.isin(top_ids), "Womens E-Mail", "No E-Mail"),
        "Random": np.where(
            d.row_id.isin(top_ids),
            np.where(np.arange(len(d)) % 2 == 0, "Mens E-Mail", "Womens E-Mail"),
            "No E-Mail"
        ),
        "No Email": np.array(["No E-Mail"] * len(d))
    }

    base_conv = d.loc[d.segment == "No E-Mail", "conversion"].mean()
    rows = []
    for name, acts in policies.items():
        actions = pd.Series(acts)
        value = ipw_policy_value(d, actions, "conversion")
        rows.append({
            "policy": name,
            "targeted": int((actions != "No E-Mail").sum()),
            "estimated_conversion_rate": value,
            "incremental_conversion_rate": value - base_conv,
            "estimated_incremental_conversions": (value - base_conv) * len(d)
        })
    return pd.DataFrame(rows)

policy_cmp = policy_table_local(test_pred, depth=0.20)
display(policy_cmp.style.format({
    "estimated_conversion_rate":"{:.4%}",
    "incremental_conversion_rate":"{:.4%}",
    "estimated_incremental_conversions":"{:.2f}"
}))
policy_cmp.to_csv(METRICS_DIR / "notebook_06_policy_comparison.csv", index=False)

## 6. Customer prediction export

In [ ]:
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
output_cols = [
    "row_id","recency","history","mens","womens","zip_code","newbie","channel","history_segment",
    "pred_conversion_mens","pred_conversion_control_mens",
    "pred_conversion_womens","pred_conversion_control_womens",
    "uplift_mens","uplift_womens","uplift_any_email",
    "max_uplift","recommended_action"
]
customer_scores = ranked[output_cols].copy()
customer_scores.to_csv(PREDICTIONS_DIR / "notebook_06_customer_scores.csv", index=False)

top_10 = customer_scores.head(10)
top_10.to_csv(PREDICTIONS_DIR / "notebook_06_top_10_customers.csv", index=False)

print("Saved:", PREDICTIONS_DIR / "notebook_06_customer_scores.csv")
display(top_10)

## 7. Final recommendation framework

The final operational decision should be based on the **held-out uplift evaluation**, not simply on the largest raw predicted conversion probability.

The recommended policy is:

**Target customers with positive estimated incremental response, rank them by expected uplift, and choose the campaign depth that produces the strongest validated incremental business outcome.**

The estimates are model-based and should be treated as decision-support outputs rather than guaranteed individual causal effects.